# Block 1 Solution — Read, Explore, and Report

Completed version of `01_read_explore/read_explore_exercise.ipynb`. The markdown notes at each
TODO explain the approach.

In [1]:
import odmlib.define_loader as DL
import odmlib.loader as LD

loader = LD.ODMLoader(DL.XMLDefineLoader(model_package="define_2_1"))
loader.open_odm_document("../data/defineV21-SDTM.xml")

odm = loader.root()                  # call root() once and keep the reference
mdv = odm.Study.MetaDataVersion      # single objects in Define-XML - no [0]

print("Study:      ", odm.Study.GlobalVariables.StudyName)
print("MetaDataVer:", mdv.Name)
print("Define ver: ", mdv.DefineVersion)

Study:       CDISC01_1
MetaDataVer: Study CDISC01_1, Data Definitions V-1
Define ver:  2.1.0


In [2]:
for std in mdv.Standards.Standard:
    print(f"{std.OID:<10} {std.Name:<10} {std.Version:<12} ({std.Type})")

STD.1      SDTMIG     3.1.2        (IG)
STD.2      SDTMIG     3.2          (IG)
STD.2_1    SDTMIG-MD  1.0          (IG)
STD.3      CDISC/NCI  2011-12-09   (CT)
STD.4      CDISC/NCI  2015-12-18   (CT)


In [3]:
for igd in mdv.ItemGroupDef:
    print(f"{igd.OID:<12} {igd.Name:<8} {len(igd.ItemRef):>3} variables   {igd.Structure}")

IG.TS        TS         6 variables   One record per trial summary parameter value
IG.DI        DI         7 variables   One record per device identifier per device
IG.DM        DM        16 variables   One record per subject
IG.EC        EC        12 variables   One record per constant dosing interval per subject
IG.EX        EX        12 variables   One record per constant dosing interval per subject
IG.LB        LB        29 variables   One record per analyte per visit per subject
IG.VS        VS        18 variables   One record per vital sign measurement per visit per subject
IG.XS        XS        18 variables   One record per finding per visit per subject
IG.XX        XX        17 variables   One record per finding per visit per subject
IG.SUPPDM    SUPPDM    10 variables   One record per IDVAR, IDVARVAL, and QNAM value per subject
IG.SUPPVS    SUPPVS    10 variables   One record per IDVAR, IDVARVAL, and QNAM value per subject


## TODO 1 — Find a variable and describe it

`find(class_name, attribute, value)` searches all descendants and returns the first match (or
`None`). One call replaces a hand-written nested loop, and it works from any element — here we
search from the `MetaDataVersion`.

In [4]:
age = mdv.find("ItemDef", "OID", "IT.DM.AGE")

print("Name:       ", age.Name)
print("DataType:   ", age.DataType)
print("Length:     ", age.Length)
print("Description:", age.Description.TranslatedText[0]._content)
print("Origin:     ", age.Origin[0].Type)

Name:        AGE
DataType:    integer
Length:      2
Description: Age
Origin:      Derived


## TODO 2 — List a dataset's variables

An `ItemRef` holds the dataset-specific facts (order, mandatory, key sequence) and points at
an `ItemDef` by OID for the variable-level facts — so a listing is a sort plus one `find()`
per reference.

In [5]:
vs = mdv.find("ItemGroupDef", "OID", "IG.VS")

print(f"{'#':>3} {'Name':<10} {'Type':<10} {'Len':>4} {'Mandatory'}")
for ref in sorted(vs.ItemRef, key=lambda r: int(r.OrderNumber)):
    item = mdv.find("ItemDef", "OID", ref.ItemOID)
    length = item.Length if item.Length is not None else ""
    print(f"{ref.OrderNumber:>3} {item.Name:<10} {item.DataType:<10} {length!s:>4} {ref.Mandatory}")

  # Name       Type        Len Mandatory
  1 STUDYID    text          7 Yes
  2 DOMAIN     text          2 Yes
  3 USUBJID    text         14 Yes
  4 VSSEQ      integer       2 Yes
  5 VSTESTCD   text         20 Yes
  6 VSTEST     text         24 Yes
  7 VSPOS      text          7 No
  8 VSORRES    text         30 No
  9 VSORRESU   text         20 No
 10 VSSTRESC   text          6 Yes
 11 VSSTRESN   float         6 No
 12 VSSTRESU   text          9 No
 13 VSBLFL     text          1 No
 14 VISITNUM   integer       2 No
 15 VISIT      text          8 No
 16 VISITDY    integer       3 No
 17 VSDTC      date            No
 18 VSDY       integer       3 No


## TODO 3 — Decode a codelist

Two reference hops: `ItemDef → CodeListRef.CodeListOID → CodeList`. This codelist uses
`CodeListItem` (coded value + decode); enumerated codelists would use `EnumeratedItem`
(coded value only), so robust code checks both.

In [6]:
vstestcd = mdv.find("ItemDef", "OID", "IT.VS.VSTESTCD")
cl = mdv.find("CodeList", "OID", vstestcd.CodeListRef.CodeListOID)

print(f"{vstestcd.Name} uses codelist {cl.OID} ({cl.Name}):")
for term in cl.CodeListItem:
    print(f"   {term.CodedValue:<10} = {term.Decode.TranslatedText[0]._content}")
for term in cl.EnumeratedItem:
    print(f"   {term.CodedValue}")

VSTESTCD uses codelist CL.VSTESTCD (Vital Signs Test Code):
   BMI        = Body Mass Index
   DIABP      = Diastolic Blood Pressure
   FRMSIZE    = Body Frame Size
   HEIGHT     = Height
   PULSE      = Pulse Rate
   SYSBP      = Systolic Blood Pressure
   WEIGHT     = Weight


## TODO 4 — Basic define.xml metrics

Every metadata collection on `MetaDataVersion` is a plain Python list, so a document profile
is just `len()` calls — no XPath, no namespace handling.

In [7]:
metrics = {
    "datasets": len(mdv.ItemGroupDef),
    "variables": len(mdv.ItemDef),
    "codelists": len(mdv.CodeList),
    "methods": len(mdv.MethodDef),
    "value lists": len(mdv.ValueListDef),
    "where clauses": len(mdv.WhereClauseDef),
    "comments": len(mdv.CommentDef),
    "documents": len(mdv.leaf),
}

for name, count in metrics.items():
    print(f"{name:>14}: {count}")

      datasets: 11
     variables: 179
     codelists: 40
       methods: 33
   value lists: 8
 where clauses: 32
      comments: 29
     documents: 3


Expected: 11 datasets, 179 variables, 40 codelists, 33 methods, 8 value lists, 32 where
clauses, 29 comments, 3 documents.

## Stretch — the same pattern, a different standard

Only the loader class changes; navigation and `find()` work exactly the same on the ARM model.

In [8]:
import odmlib.arm_loader as AL

arm_loader = LD.ODMLoader(AL.XMLArmLoader())
arm_loader.open_odm_document("../data/definev21-adam.xml")
arm_mdv = arm_loader.root().Study.MetaDataVersion

for rd in arm_mdv.AnalysisResultDisplays.ResultDisplay:
    print(rd.OID)
    print("   ", str(rd.Description.TranslatedText[0]))
    for ar in rd.AnalysisResult:
        print("     result:", ar.OID)

RD.Table_14-3.01
    Primary Endpoint Analysis: ADAS-Cog - Summary at Week 24 - LOCF (Efficacy Population)
     result: AR.Table_14-3.01.R.1
     result: AR.Table_14-3.01.R.2
RD.Table_14-5.02
    Incidence of Treatment Emergent Serious Adverse Events by Treatment Group
     result: AR.Table_14-5.02.R.1
